# ***Cell 1 Imports & Configuration***

In [ ]:
from google.colab import drive
import os
import json
import numpy as np
import pandas as pd

from scipy.interpolate import interp1d
from sklearn.preprocessing import StandardScaler

drive.mount('/content/drive')

RANDOM_SEED = 42
DATASET_VERSION = "V1_FROZEN"

np.random.seed(RANDOM_SEED)

OUTPUT_DIR = f"/content/dataset_{DATASET_VERSION.lower()}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

DRIVE_ZIP = "/content/drive/MyDrive/cleaned_dataset.zip"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# ***Cell 2 Unzip Dataset***

In [ ]:
if not os.path.exists(DRIVE_ZIP):
    raise FileNotFoundError(f"Dataset not found: {DRIVE_ZIP}")

!unzip -o -q "/content/drive/MyDrive/cleaned_dataset.zip" -d /content/

if os.path.exists("/content/cleaned_dataset"):
    DATASET_DIR = "/content/cleaned_dataset"
else:
    DATASET_DIR = "/content"

print("Dataset Directory:", DATASET_DIR)

Dataset Directory: /content/cleaned_dataset


# ***Cell 3 Locate Metadata and Data Folder***

In [ ]:
METADATA_PATH = os.path.join(DATASET_DIR, "metadata.csv")
DATA_CSV_DIR = os.path.join(DATASET_DIR, "data")

if not os.path.exists(METADATA_PATH):
    raise FileNotFoundError("metadata.csv not found")

if not os.path.exists(DATA_CSV_DIR):
    raise FileNotFoundError("data folder not found")

print("Metadata:", METADATA_PATH)
print("Data Directory:", DATA_CSV_DIR)

Metadata: /content/cleaned_dataset/metadata.csv
Data Directory: /content/cleaned_dataset/data


# ***Cell 4 Load and Prepare Metadata***

In [ ]:
df = pd.read_csv(METADATA_PATH)

df["Capacity"] = pd.to_numeric(
    df["Capacity"].astype(str).str.replace(r"[\[\]]", "", regex=True),
    errors="coerce"
)

dis_df = df[
    (df["type"] == "discharge") &
    (df["Capacity"] > 0.05)
].sort_values(
    ["battery_id", "test_id"]
).copy()

print("Total metadata rows:", len(df))
print("Discharge cycles:", len(dis_df))
print("Unique batteries:", dis_df["battery_id"].nunique())

Total metadata rows: 7565
Discharge cycles: 2714
Unique batteries: 34


# ***Cell 5 Calculate SOH and RUL***

In [ ]:
c0_map = (
    dis_df
    .groupby("battery_id")["Capacity"]
    .first()
    .to_dict()
)

dis_df["initial_capacity"] = dis_df["battery_id"].map(c0_map)
dis_df["SOH"] = dis_df["Capacity"] / dis_df["initial_capacity"]

In [ ]:
def assign_rul(sub):
    eol_thresh = 0.70 * sub["initial_capacity"].iloc[0]

    eol_sub = sub[sub["Capacity"] <= eol_thresh]

    if len(eol_sub) > 0:
        eol_cycle = eol_sub["test_id"].iloc[0]
    else:
        eol_cycle = sub["test_id"].max()

    sub = sub.copy()
    sub["EOL_cycle"] = eol_cycle
    sub["RUL"] = np.maximum(
        0,
        eol_cycle - sub["test_id"]
    )

    return sub


dis_df = (
    dis_df
    .groupby("battery_id", group_keys=False)
    .apply(assign_rul)
)

/tmp/ipykernel_2743/1736434174.py:24: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(assign_rul)


# ***Cell 6 Define Time Grid***

In [ ]:
GRID_DT = 10
GRID_MAX_T = 500

TIME_GRID = np.arange(
    0,
    GRID_MAX_T + GRID_DT,
    GRID_DT
)

GRID_LEN = len(TIME_GRID)

CHANNELS = [
    "Voltage_measured",
    "Current_measured",
    "Temperature_measured"
]

print("Time steps:", GRID_LEN)
print("Channels:", CHANNELS)

Time steps: 51
Channels: ['Voltage_measured', 'Current_measured', 'Temperature_measured']


# ***Cell 7 Align All Battery Cycles***

In [ ]:
aligned_cycles = []
meta_records = []

for _, r in dis_df.iterrows():

    fpath = os.path.join(
        DATA_CSV_DIR,
        r["filename"]
    )

    if not os.path.exists(fpath):
        continue

    try:
        raw = pd.read_csv(fpath)

        if (
            len(raw) < 10 or
            "Time" not in raw.columns or
            raw["Time"].iloc[-1] < 60
        ):
            continue

        raw = (
            raw
            .drop_duplicates(subset=["Time"])
            .sort_values("Time")
        )

        t_raw = raw["Time"].values

        cycle_tensor = np.zeros(
            (GRID_LEN, len(CHANNELS)),
            dtype=np.float32
        )

        valid = True

        for c_idx, col in enumerate(CHANNELS):

            if col not in raw.columns:
                valid = False
                break

            interp_fn = interp1d(
                t_raw,
                raw[col].values,
                kind="linear",
                bounds_error=False,
                fill_value="extrapolate"
            )

            cycle_tensor[:, c_idx] = interp_fn(
                TIME_GRID
            )

        if not valid:
            continue

        aligned_cycles.append(cycle_tensor)

        meta_records.append({
            "battery_id": r["battery_id"],
            "test_id": int(r["test_id"]),
            "ambient_temperature": r["ambient_temperature"],
            "Capacity": float(r["Capacity"]),
            "SOH": float(r["SOH"]),
            "RUL": int(r["RUL"]),
            "filename": r["filename"]
        })

    except Exception:
        continue

# ***Cell 8 Create Aligned Dataset***

In [ ]:
if len(aligned_cycles) == 0:
    raise RuntimeError("No valid cycles were processed.")

aligned_X = np.stack(
    aligned_cycles,
    axis=0
)

meta_df = pd.DataFrame(
    meta_records
).reset_index(drop=True)

print("Aligned X shape:", aligned_X.shape)
print("Metadata shape:", meta_df.shape)

Aligned X shape: (2705, 51, 3)
Metadata shape: (2705, 7)


# ***Cell 9 Define Cell-Level Train/Validation/Test Split***

In [ ]:
test_cells = [
    "B0018", "B0028", "B0032",
    "B0036", "B0040", "B0043",
    "B0044", "B0051", "B0056"
]

val_cells = [
    "B0007", "B0027", "B0031",
    "B0034", "B0039", "B0042",
    "B0048", "B0050", "B0054"
]

train_cells = [
    b for b in meta_df["battery_id"].unique()
    if b not in test_cells
    and b not in val_cells
]

print("Train cells:", len(train_cells))
print("Validation cells:", len(val_cells))
print("Test cells:", len(test_cells))

Train cells: 16
Validation cells: 9
Test cells: 9


# ***Cell 10 Leakage-Safe Normalization***

In [ ]:
train_mask = meta_df["battery_id"].isin(
    train_cells
).values

scaler = StandardScaler()

scaler.fit(
    aligned_X[train_mask]
    .reshape(-1, len(CHANNELS))
)

aligned_X_scaled = np.zeros_like(
    aligned_X
)

for i in range(len(aligned_X)):
    aligned_X_scaled[i] = scaler.transform(
        aligned_X[i]
    )

print("Normalization completed.")
print("Scaler means:", scaler.mean_)
print("Scaler scales:", scaler.scale_)

Normalization completed.
Scaler means: [ 3.67462081 -2.17975817 22.61929321]
Scaler scales: [ 0.26836292  1.1944407  13.47057353]


# ***Cell 11 Create Sliding Windows***

In [ ]:
WINDOW_SIZE = 10

windowed_X = []
windowed_y_soh = []
windowed_y_rul = []
windowed_y_cap = []
windowed_meta = []

for b_id, grp in meta_df.groupby("battery_id"):

    grp_indices = grp.index.values

    if len(grp_indices) < WINDOW_SIZE:
        continue

    for start in range(
        len(grp_indices) - WINDOW_SIZE + 1
    ):

        end = start + WINDOW_SIZE
        target_idx = grp_indices[end - 1]

        windowed_X.append(
            aligned_X_scaled[
                grp_indices[start:end]
            ]
        )

        windowed_y_soh.append(
            meta_df.loc[target_idx, "SOH"]
        )

        windowed_y_rul.append(
            meta_df.loc[target_idx, "RUL"]
        )

        windowed_y_cap.append(
            meta_df.loc[target_idx, "Capacity"]
        )

        windowed_meta.append({
            "battery_id": b_id,
            "target_test_id": int(
                meta_df.loc[target_idx, "test_id"]
            ),
            "ambient_temperature":
                meta_df.loc[
                    target_idx,
                    "ambient_temperature"
                ],
            "window_start_idx": start,
            "window_end_idx": end - 1
        })

# ***Cell 12 Convert Windows to NumPy Arrays***

In [ ]:
windowed_X = np.stack(
    windowed_X,
    axis=0
)

windowed_y_soh = np.array(
    windowed_y_soh,
    dtype=np.float32
)

windowed_y_rul = np.array(
    windowed_y_rul,
    dtype=np.float32
)

windowed_y_cap = np.array(
    windowed_y_cap,
    dtype=np.float32
)

windowed_meta_df = pd.DataFrame(
    windowed_meta
).reset_index(drop=True)

print("Windowed X:", windowed_X.shape)
print("SOH:", windowed_y_soh.shape)
print("RUL:", windowed_y_rul.shape)
print("Capacity:", windowed_y_cap.shape)

Windowed X: (2404, 10, 51, 3)
SOH: (2404,)
RUL: (2404,)
Capacity: (2404,)


# ***Cell 13 Create Train/Validation/Test Windows***


In [ ]:
win_train_mask = (
    windowed_meta_df["battery_id"]
    .isin(train_cells)
    .values
)

win_val_mask = (
    windowed_meta_df["battery_id"]
    .isin(val_cells)
    .values
)

win_test_mask = (
    windowed_meta_df["battery_id"]
    .isin(test_cells)
    .values
)

# ***Cell 14 Leakage and Data Integrity Checks***

In [ ]:
assert len(set(train_cells) & set(val_cells)) == 0
assert len(set(train_cells) & set(test_cells)) == 0
assert len(set(val_cells) & set(test_cells)) == 0

assert not np.isnan(windowed_X).any()
assert not np.isnan(windowed_y_soh).any()
assert not np.isnan(windowed_y_rul).any()

assert (
    set(windowed_meta_df.loc[
        win_train_mask, "battery_id"
    ]).issubset(set(train_cells))
)

assert (
    set(windowed_meta_df.loc[
        win_val_mask, "battery_id"
    ]).issubset(set(val_cells))
)

assert (
    set(windowed_meta_df.loc[
        win_test_mask, "battery_id"
    ]).issubset(set(test_cells))
)

print("All leakage and integrity checks PASSED.")

All leakage and integrity checks PASSED.


# ***Cell 15 Save Train/Validation/Test Dataset***

In [ ]:
np.savez_compressed(
    os.path.join(
        OUTPUT_DIR,
        "dataset_v1_cell_splits.npz"
    ),
    X_train=windowed_X[win_train_mask],
    y_train_soh=windowed_y_soh[win_train_mask],
    y_train_rul=windowed_y_rul[win_train_mask],
    X_val=windowed_X[win_val_mask],
    y_val_soh=windowed_y_soh[win_val_mask],
    y_val_rul=windowed_y_rul[win_val_mask],
    X_test=windowed_X[win_test_mask],
    y_test_soh=windowed_y_soh[win_test_mask],
    y_test_rul=windowed_y_rul[win_test_mask]
)

print("Cell split dataset saved.")

Cell split dataset saved.


# ***Cell 16 Save Full Window Dataset***

In [ ]:
np.savez_compressed(
    os.path.join(
        OUTPUT_DIR,
        "dataset_v1_full_windows.npz"
    ),
    X=windowed_X,
    y_soh=windowed_y_soh,
    y_rul=windowed_y_rul,
    y_cap=windowed_y_cap
)

print("Full window dataset saved.")

Full window dataset saved.


# ***Cell 17 Save Metadata***

In [ ]:
meta_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "dataset_v1_cycles_metadata.csv"
    ),
    index=False
)

windowed_meta_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "dataset_v1_windows_metadata.csv"
    ),
    index=False
)

print("Metadata files saved.")

Metadata files saved.


# ***Cell 18 Create Dataset Manifest***

In [ ]:
manifest = {
    "status": "APPROVED_AND_FROZEN",
    "dataset_version": DATASET_VERSION,
    "timestamp_frozen": pd.Timestamp.now().isoformat(),
    "random_seed": RANDOM_SEED,

    "defense_parameters": {
        "observation_window_seconds": GRID_MAX_T,
        "sampling_dt_seconds": GRID_DT,
        "time_steps_per_cycle": GRID_LEN,
        "channels": CHANNELS,
        "sliding_window_cycles": WINDOW_SIZE,
        "normalization_defense":
            "StandardScaler fit exclusively on training cells",
        "rul_definition":
            "Remaining cycles until capacity drops below 70% of initial C0",
        "soh_definition":
            "Capacity_k / Initial_Capacity_C0"
    },

    "shapes": {
        "full_windowed_tensor":
            list(windowed_X.shape),
        "train_windows":
            int(win_train_mask.sum()),
        "val_windows":
            int(win_val_mask.sum()),
        "test_windows":
            int(win_test_mask.sum())
    },

    "cell_split_distribution": {
        "train_cells": sorted(train_cells),
        "val_cells": sorted(val_cells),
        "test_cells": sorted(test_cells)
    },

    "scaler_statistics": {
        "channel_means":
            scaler.mean_.tolist(),
        "channel_scales":
            scaler.scale_.tolist()
    }
}

with open(
    os.path.join(
        OUTPUT_DIR,
        "dataset_v1_manifest.json"
    ),
    "w"
) as f:
    json.dump(
        manifest,
        f,
        indent=4
    )

print("Manifest created.")

Manifest created.


# ***Cell 19 Final Dataset Freeze Summary***

In [ ]:
print("=" * 60)
print("DATASET VERSION 1 FREEZE SUMMARY")
print("=" * 60)

print(f"Status: {manifest['status']}")
print(f"Total Cycles Processed: {len(aligned_X)}")
print(
    f"Total Sliding Windows (W={WINDOW_SIZE}): "
    f"{len(windowed_X)}"
)

print(
    "Tensor Shape: "
    f"(Windows={windowed_X.shape[0]}, "
    f"Cycles={windowed_X.shape[1]}, "
    f"Steps={windowed_X.shape[2]}, "
    f"Channels={windowed_X.shape[3]})"
)

print(
    f"Train Windows: {win_train_mask.sum()} "
    f"({len(train_cells)} cells)"
)

print(
    f"Val Windows:   {win_val_mask.sum()} "
    f"({len(val_cells)} cells)"
)

print(
    f"Test Windows:  {win_test_mask.sum()} "
    f"({len(test_cells)} cells)"
)

print("Leakage Checks: PASSED")
print("Normalization: Training cells only")
print(f"Output Directory: {OUTPUT_DIR}")
print("=" * 60)

DATASET VERSION 1 FREEZE SUMMARY
Status: APPROVED_AND_FROZEN
Total Cycles Processed: 2705
Total Sliding Windows (W=10): 2404
Tensor Shape: (Windows=2404, Cycles=10, Steps=51, Channels=3)
Train Windows: 1002 (16 cells)
Val Windows:   697 (9 cells)
Test Windows:  705 (9 cells)
Leakage Checks: PASSED
Normalization: Training cells only
Output Directory: /content/dataset_v1_frozen


# ***Cell 20 Verify Saved Files***

In [ ]:
print("Generated files:")

for file in sorted(os.listdir(OUTPUT_DIR)):
    path = os.path.join(OUTPUT_DIR, file)
    print(
        f"{file:<45} "
        f"{os.path.getsize(path) / (1024**2):.2f} MB"
    )

Generated files:
dataset_v1_cell_splits.npz                    1.51 MB
dataset_v1_cycles_metadata.csv                0.16 MB
dataset_v1_full_windows.npz                   1.52 MB
dataset_v1_manifest.json                      0.00 MB
dataset_v1_windows_metadata.csv               0.04 MB
